# 과제 - 신경망을 이용한 손글씨 숫자 인식



## 1. 환경설정



In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요 (저장소 클론 후 경로·모듈 로드)
# public 저장소는 토큰 없이 Enter, private 저장소만 Personal Access Token을 입력하세요.
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from getpass import getpass

    default_git_url = "github.com/YangSiJun528/WEEK13_mnist-lab.git"
    git_url = input(f"GitHub 저장소 URL (기본: {default_git_url}): ").strip() or default_git_url
    git_url = git_url.replace("https://", "").replace("http://", "")
    git_branch = input("브랜치명 (기본: temp): ").strip() or "temp"
    token = getpass("GitHub Personal Access Token (public 저장소면 Enter): ").strip()

    # URL 마지막 경로를 저장소 폴더명으로 사용합니다. (예: .../mnist-lab.git -> mnist-lab)
    repo_name = Path(git_url.rstrip("/")).name
    if repo_name.endswith(".git"):
        repo_name = repo_name[:-4]

    clone_url = f"https://{token}@{git_url}" if token else f"https://{git_url}"
    if not Path(repo_name).exists():
        subprocess.run(["git", "clone", "-b", git_branch, clone_url], check=True)
    else:
        print(f"이미 {repo_name} 폴더가 있어 clone을 건너뜁니다.")

    os.chdir(repo_name)
    sys.path.insert(0, str(Path.cwd() / "src"))
    print("실행 경로:", Path.cwd())
else:
    sys.path.insert(0, "./src")
    print("실행 경로:", Path.cwd())


## 2. 데이터 로드

In [ ]:
from data import load_mnist

(x_train, y_train), (x_test, y_test) = load_mnist()
print('Train:', x_train.shape, y_train.shape)
print('Test:', x_test.shape, y_test.shape)

## 3. 구현 및 테스트 통과 확인

`src/` 아래 역할별 파일의 **TODO**를 순서대로 구현한 뒤 아래 셀을 실행하세요.
- 주요 구현 파일: `activations.py`, `layers.py`, `losses.py`, `optimizers.py`, `network.py`, `training.py`
- 구현 파일은 역할별 모듈을 직접 import합니다. 예: `from network import NeuralNetwork`
- 개발 순서: 과제 안내문 참조
- 테스트: `tests/` 아래의 단계별 단위 테스트를 필요한 파일부터 실행합니다. 처음에는 전체 테스트보다 맡은 부분의 테스트 파일을 먼저 실행하세요.
    - ReLU만 확인: `TEST_TARGET = "tests/test_relu.py"`
    - 파일 안의 일부 테스트만 확인: `PYTEST_KEYWORD = "backward"`
    - 전체 테스트 확인: `TEST_TARGET = "tests/"`

In [ ]:
import subprocess
import sys
from pathlib import Path

# Colab/로컬 모두 현재 노트북 실행 위치를 저장소 루트로 사용합니다.
repo_dir = Path.cwd()

# 처음에는 자신이 구현 중인 부분의 테스트 파일만 실행하세요.
# 예: tests/test_relu.py, tests/test_affine.py, tests/test_training.py
TEST_TARGET = "tests/test_relu.py"

# 특정 이름이 들어간 테스트만 실행하고 싶을 때 사용합니다.
# 예: "backward". 전체 파일을 실행하려면 빈 문자열로 둡니다.
PYTEST_KEYWORD = ""

cmd = [sys.executable, "-m", "pytest", TEST_TARGET, "-v"]
if PYTEST_KEYWORD:
    cmd.extend(["-k", PYTEST_KEYWORD])

print("실행 경로:", repo_dir)
print("실행 명령:", " ".join(cmd))
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    cwd=str(repo_dir)
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode == 0:
    print("\n선택한 테스트를 통과했습니다.")
else:
    print("\n선택한 테스트 중 실패가 있습니다.")


## 4. 모델·옵티마이저 생성 및 학습

In [ ]:
from network import NeuralNetwork
from optimizers import Adam
from training import train

model = NeuralNetwork(use_batchnorm=True, use_dropout=True)  # BatchNorm, Dropout 필수
optimizer = Adam(lr=0.001)

loss_history = train(model, optimizer, x_train, y_train, epochs=20, batch_size=128)

## 5. 평가 및 손실 커브

In [ ]:
from training import evaluate, plot_loss_history

acc, n_params = evaluate(model, x_test, y_test)
print(f'Test Accuracy: {acc:.2f}%')
print(f'Total Params: {n_params:,}')

plot_loss_history(loss_history)

## 6. 비교 실험: Optimizer, BatchNorm/Dropout, Learning Rate

아래 셀은 ReLU와 He 초기화를 고정하고 6개 전략을 실행합니다.
- SGD vs Adam(ADRM): `sgd_lr_0_01` vs `adam_baseline`
- BatchNorm/Dropout 유무: `adam_baseline` vs `no_batchnorm`, `adam_baseline` vs `no_dropout`
- 학습률 비교: `adam_baseline` vs `adam_lr_0_01` vs `adam_lr_decay`

실행 결과는 `experiment_logs/`에 txt/csv/md로 저장되고, 비교 그래프는 `report_assets/`에 svg로 저장됩니다.


In [ ]:
import csv
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from losses import cross_entropy_loss
from network import NeuralNetwork
from optimizers import Adam, SGD

EPOCHS = 20
BATCH_SIZE = 128
SEED = 42
TRAIN_EVAL_SIZE = 10000

LOG_DIR = Path("experiment_logs")
ASSET_DIR = Path("report_assets")
LOG_DIR.mkdir(exist_ok=True)
ASSET_DIR.mkdir(exist_ok=True)

EXPERIMENT_CONFIGS = [
    {
        "id": "adam_baseline",
        "label": "Adam lr=0.001 + BN + Dropout",
        "groups": ["optimizer", "regularization", "learning_rate"],
        "optimizer": "Adam",
        "lr": 0.001,
        "use_batchnorm": True,
        "use_dropout": True,
        "dropout_ratio": 0.5,
    },
    {
        "id": "sgd_lr_0_01",
        "label": "SGD lr=0.01 + BN + Dropout",
        "groups": ["optimizer"],
        "optimizer": "SGD",
        "lr": 0.01,
        "use_batchnorm": True,
        "use_dropout": True,
        "dropout_ratio": 0.5,
    },
    {
        "id": "no_batchnorm",
        "label": "Adam lr=0.001 + no BN + Dropout",
        "groups": ["regularization"],
        "optimizer": "Adam",
        "lr": 0.001,
        "use_batchnorm": False,
        "use_dropout": True,
        "dropout_ratio": 0.5,
    },
    {
        "id": "no_dropout",
        "label": "Adam lr=0.001 + BN + no Dropout",
        "groups": ["regularization"],
        "optimizer": "Adam",
        "lr": 0.001,
        "use_batchnorm": True,
        "use_dropout": False,
        "dropout_ratio": 0.0,
    },
    {
        "id": "adam_lr_0_01",
        "label": "Adam lr=0.01 + BN + Dropout",
        "groups": ["learning_rate"],
        "optimizer": "Adam",
        "lr": 0.01,
        "use_batchnorm": True,
        "use_dropout": True,
        "dropout_ratio": 0.5,
    },
    {
        "id": "adam_lr_decay",
        "label": "Adam lr decay 0.01*0.6^epoch + BN + Dropout",
        "groups": ["learning_rate"],
        "optimizer": "Adam",
        "lr": 0.01,
        "lr_schedule": lambda epoch_index: 0.01 * (0.6 ** epoch_index),
        "use_batchnorm": True,
        "use_dropout": True,
        "dropout_ratio": 0.5,
    },
]

GROUPS = {
    "optimizer": {
        "title": "SGD vs Adam(ADRM)",
        "ids": ["sgd_lr_0_01", "adam_baseline"],
        "asset_prefix": "optimizer_sgd_vs_adam",
    },
    "regularization": {
        "title": "BatchNorm / Dropout 유무 비교",
        "ids": ["adam_baseline", "no_batchnorm", "no_dropout"],
        "asset_prefix": "regularization_bn_dropout",
    },
    "learning_rate": {
        "title": "학습률 비교",
        "ids": ["adam_baseline", "adam_lr_0_01", "adam_lr_decay"],
        "asset_prefix": "learning_rate_comparison",
    },
}


def build_model(config):
    return NeuralNetwork(
        use_batchnorm=config["use_batchnorm"],
        use_dropout=config["use_dropout"],
        dropout_ratio=config["dropout_ratio"],
        init_method="he",  # ReLU 실험 기준에서는 He를 고정합니다.
    )


def build_optimizer(config):
    if config["optimizer"] == "SGD":
        return SGD(lr=config["lr"])
    if config["optimizer"] == "Adam":
        return Adam(lr=config["lr"])
    raise ValueError(f"지원하지 않는 optimizer: {config['optimizer']}")


def accuracy(y_pred, y_true):
    return float(np.mean(np.argmax(y_pred, axis=1) == y_true) * 100)


def evaluate_metrics(model, x, y):
    y_pred = model.predict(x)
    return {
        "loss": float(cross_entropy_loss(y_pred, y)),
        "acc": accuracy(y_pred, y),
    }


def train_one_epoch(model, optimizer, x_train, y_train, batch_size):
    train_size = x_train.shape[0]
    indices = np.random.permutation(train_size)
    epoch_loss = 0.0
    batch_count = 0

    for start in range(0, train_size, batch_size):
        batch_indices = indices[start:start + batch_size]
        x_batch = x_train[batch_indices]
        y_batch = y_train[batch_indices]

        y_pred = model.forward(x_batch, train=True)
        loss = cross_entropy_loss(y_pred, y_batch)

        dout = y_pred.copy()
        dout[np.arange(x_batch.shape[0]), y_batch] -= 1
        dout /= x_batch.shape[0]

        model.backward(dout)
        optimizer.update(model.params, model.grads)

        epoch_loss += loss
        batch_count += 1

    return float(epoch_loss / batch_count)


def run_strategy_experiments(configs, x_train, y_train, x_test, y_test):
    rng = np.random.default_rng(SEED)
    eval_size = min(TRAIN_EVAL_SIZE, x_train.shape[0])
    eval_indices = rng.choice(x_train.shape[0], size=eval_size, replace=False)
    x_train_eval = x_train[eval_indices]
    y_train_eval = y_train[eval_indices]

    results = []
    log_lines = []

    for config_index, config in enumerate(configs):
        np.random.seed(SEED + config_index)
        model = build_model(config)
        optimizer = build_optimizer(config)
        params = int(sum(param.size for param in model.params.values()))
        history = []

        header = f"[{config['id']}] {config['label']}"
        print("\n" + header)
        log_lines.append(header)
        log_lines.append(
            "config: "
            f"optimizer={config['optimizer']}, lr={config['lr']}, "
            f"batchnorm={config['use_batchnorm']}, dropout={config['use_dropout']}, "
            f"dropout_ratio={config['dropout_ratio']}, init=he, params={params}"
        )

        for epoch_index in range(EPOCHS):
            if "lr_schedule" in config:
                optimizer.lr = float(config["lr_schedule"](epoch_index))

            train_loss = train_one_epoch(model, optimizer, x_train, y_train, BATCH_SIZE)
            train_metrics = evaluate_metrics(model, x_train_eval, y_train_eval)
            val_metrics = evaluate_metrics(model, x_test, y_test)

            record = {
                "strategy": config["id"],
                "label": config["label"],
                "optimizer": config["optimizer"],
                "epoch": epoch_index + 1,
                "lr": float(optimizer.lr),
                "train_loss": train_loss,
                "train_acc": train_metrics["acc"],
                "val_loss": val_metrics["loss"],
                "val_acc": val_metrics["acc"],
                "params": params,
                "use_batchnorm": config["use_batchnorm"],
                "use_dropout": config["use_dropout"],
                "dropout_ratio": config["dropout_ratio"],
                "init_method": "he",
            }
            history.append(record)

            line = (
                f"epoch {epoch_index + 1:02d}/{EPOCHS} "
                f"lr={optimizer.lr:.6f} "
                f"train_loss={train_loss:.4f} "
                f"train_acc={train_metrics['acc']:.2f}% "
                f"val_loss={val_metrics['loss']:.4f} "
                f"val_acc={val_metrics['acc']:.2f}% "
                f"params={params}"
            )
            print(line)
            log_lines.append(line)

        results.append({"config": config, "history": history, "model": model, "params": params})
        log_lines.append("")

    return results, log_lines


def summarize_results(results):
    summary = []
    for result in results:
        config = result["config"]
        final = result["history"][-1]
        best = max(result["history"], key=lambda row: row["val_acc"])
        summary.append(
            {
                "strategy": config["id"],
                "label": config["label"],
                "optimizer": config["optimizer"],
                "final_val_acc": final["val_acc"],
                "best_val_acc": best["val_acc"],
                "best_epoch": best["epoch"],
                "final_train_acc": final["train_acc"],
                "final_val_loss": final["val_loss"],
                "final_lr": final["lr"],
                "params": result["params"],
                "use_batchnorm": config["use_batchnorm"],
                "use_dropout": config["use_dropout"],
            }
        )
    return summary


def markdown_table(rows):
    lines = [
        "| strategy | optimizer | final val acc | best val acc | best epoch | train acc | val loss | final lr | params | BN | Dropout |",
        "| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | --- | --- |",
    ]
    for row in rows:
        lines.append(
            f"| {row['strategy']} | {row['optimizer']} | {row['final_val_acc']:.2f}% | "
            f"{row['best_val_acc']:.2f}% | {row['best_epoch']} | {row['final_train_acc']:.2f}% | "
            f"{row['final_val_loss']:.4f} | {row['final_lr']:.6f} | {row['params']:,} | "
            f"{row['use_batchnorm']} | {row['use_dropout']} |"
        )
    return "\n".join(lines)


def save_logs(results, summary, log_lines):
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    txt_path = LOG_DIR / f"grouped_strategy_run_{timestamp}.txt"
    csv_path = LOG_DIR / f"grouped_strategy_run_{timestamp}.csv"
    md_path = LOG_DIR / f"grouped_strategy_summary_{timestamp}.md"

    txt_path.write_text("\n".join(log_lines), encoding="utf-8")

    fieldnames = [
        "strategy", "label", "optimizer", "epoch", "lr",
        "train_loss", "train_acc", "val_loss", "val_acc", "params",
        "use_batchnorm", "use_dropout", "dropout_ratio", "init_method",
    ]
    with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        for result in results:
            for row in result["history"]:
                writer.writerow(row)

    md_lines = [
        "# Grouped Strategy Run Summary",
        "",
        f"- created_at: {timestamp}",
        f"- epochs: {EPOCHS}",
        f"- batch_size: {BATCH_SIZE}",
        f"- seed: {SEED}",
        f"- train_eval_size: {TRAIN_EVAL_SIZE}",
        "- fixed activation: ReLU",
        "- fixed init: He",
        "",
        markdown_table(summary),
        "",
    ]
    md_path.write_text("\n".join(md_lines), encoding="utf-8")

    print("\n저장된 로그 파일")
    print("txt:", txt_path)
    print("csv:", csv_path)
    print("summary md:", md_path)
    return txt_path, csv_path, md_path


def select_results(results, ids):
    by_id = {result["config"]["id"]: result for result in results}
    return [by_id[strategy_id] for strategy_id in ids]


def set_padded_ylim(ax, values, lower=None, upper=None):
    values = [float(value) for value in values]
    lo = min(values)
    hi = max(values)
    span = hi - lo
    if span == 0:
        span = max(abs(hi) * 0.1, 1e-3)
    lo -= span * 0.18
    hi += span * 0.18
    if lower is not None:
        lo = max(lower, lo)
    if upper is not None:
        hi = min(upper, hi)
    ax.set_ylim(lo, hi)


def plot_metric_group(results, group_key, metric, ylabel, filename_suffix, lower=None, upper=None):
    group = GROUPS[group_key]
    selected = select_results(results, group["ids"])
    fig, ax = plt.subplots(figsize=(9, 4.8))
    all_values = []

    for result in selected:
        history = result["history"]
        epochs = [row["epoch"] for row in history]
        values = [row[metric] for row in history]
        all_values.extend(values)
        ax.plot(epochs, values, marker="o", linewidth=2, label=result["config"]["id"])

    ax.set_title(f"{group['title']} - {ylabel}")
    ax.set_xlabel("epoch")
    ax.set_ylabel(ylabel)
    set_padded_ylim(ax, all_values, lower=lower, upper=upper)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    output_path = ASSET_DIR / f"{group['asset_prefix']}_{filename_suffix}.svg"
    fig.savefig(output_path, format="svg")
    plt.show()
    print("saved:", output_path)
    return output_path


def plot_gap_group(results):
    group = GROUPS["regularization"]
    selected = select_results(results, group["ids"])
    names = [result["config"]["id"] for result in selected]
    gaps = [result["history"][-1]["train_acc"] - result["history"][-1]["val_acc"] for result in selected]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(names, gaps)
    ax.set_title("BatchNorm / Dropout 유무 비교 - final train-val acc gap")
    ax.set_ylabel("accuracy gap (%p)")
    ax.grid(True, axis="y", alpha=0.3)
    for index, gap in enumerate(gaps):
        ax.text(index, gap, f"{gap:.2f}", ha="center", va="bottom")
    fig.tight_layout()

    output_path = ASSET_DIR / "regularization_bn_dropout_train_val_gap.svg"
    fig.savefig(output_path, format="svg")
    plt.show()
    print("saved:", output_path)
    return output_path


def plot_lr_schedule(results):
    group = GROUPS["learning_rate"]
    selected = select_results(results, group["ids"])
    fig, ax = plt.subplots(figsize=(9, 4.5))

    for result in selected:
        history = result["history"]
        ax.plot(
            [row["epoch"] for row in history],
            [row["lr"] for row in history],
            marker="o",
            linewidth=2,
            label=result["config"]["id"],
        )

    ax.set_title("학습률 비교 - learning rate schedule")
    ax.set_xlabel("epoch")
    ax.set_ylabel("learning rate")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    output_path = ASSET_DIR / "learning_rate_comparison_lr_schedule.svg"
    fig.savefig(output_path, format="svg")
    plt.show()
    print("saved:", output_path)
    return output_path


def print_group_tables(summary):
    by_strategy = {row["strategy"]: row for row in summary}
    print("\n전체 비교 수치표")
    print(markdown_table(summary))

    for group_key, group in GROUPS.items():
        rows = [by_strategy[strategy_id] for strategy_id in group["ids"]]
        print(f"\n[{group['title']}]")
        print(markdown_table(rows))


experiment_results, experiment_log_lines = run_strategy_experiments(
    EXPERIMENT_CONFIGS,
    x_train,
    y_train,
    x_test,
    y_test,
)

experiment_summary = summarize_results(experiment_results)
print_group_tables(experiment_summary)
save_logs(experiment_results, experiment_summary, experiment_log_lines)

# 그룹별 비교 시각화. 전체 비교는 리포트에서 수치표로만 다룹니다.
for key in ["optimizer", "regularization", "learning_rate"]:
    plot_metric_group(experiment_results, key, "val_acc", "validation accuracy (%)", "val_accuracy", lower=90, upper=100)
    plot_metric_group(experiment_results, key, "val_loss", "validation loss", "val_loss", lower=0)

plot_gap_group(experiment_results)
plot_lr_schedule(experiment_results)
